In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
print("--- Dataset Summary ---")
for dirname, _, filenames in os.walk('/kaggle/input'):
    if len(filenames) > 0:
        print(f"Folder: {dirname} | Contains {len(filenames)} files")
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

--- Dataset Summary ---
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/test_set/test_set | Contains 4000 files
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/train/train/SeaLake | Contains 2600 files
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/train/train/Highway | Contains 2100 files
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/train/train/River | Contains 2100 files
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/train/train/Pasture | Contains 1600 files
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/train/train/Industrial | Contains 2100 files
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/train/train/Residential | Contains 2600 files
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/train/train/PermanentCrop | Contains 2100 files
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/train/train/AnnualCrop | Contains 2600 files
Folder: /kaggle/input/datasets/sakshamgarg2323/eurosat/train/train/Forest | Contains 2600 f

In [2]:
import torch

if torch.cuda.is_available():
    print(f"GPU Connected: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not detected. Please check Accelerator settings.")

GPU Connected: Tesla T4


In [3]:
import os
import sys
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import v2
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [4]:
# setting up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# defining the class mappings, could have been done simply by using image folders but forgot to do that
Class_Mapping = {
    'AnnualCrop': 0, 'Forest': 1, 'HerbaceousVegetation': 2,
    'Highway': 3, 'Industrial': 4, 'Pasture': 5,
    'PermanentCrop': 6, 'Residential': 7, 'River': 8, 'SeaLake': 9
}
# 1. Update the base path variable
base_path = '/kaggle/input/datasets/sakshamgarg2323/eurosat'

train_dir = os.path.join(base_path, 'train', 'train')
test_dir = os.path.join(base_path, 'test_set', 'test_set')



Using device: cuda


In [6]:
class EuroSATLabeled(torch.utils.data.Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms
        self.data = []
        for class_name in os.listdir(self.img_dir):
            class_path = os.path.join(self.img_dir, class_name)
            if os.path.isdir(class_path):
                class_label = Class_Mapping[class_name]
                for img_name in os.listdir(class_path):
                      self.data.append((os.path.join(class_path, img_name), class_label, img_name))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_path, class_label, img_filename = self.data[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transforms:
            img = self.transforms(img)
        return img, class_label, img_filename

class EuroSATUnlabeled(torch.utils.data.Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms
        self.data = []
        for img_name in os.listdir(self.img_dir):
              self.data.append((os.path.join(self.img_dir, img_name), img_name))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_path, img_filename = self.data[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transforms:
            img = self.transforms(img)
        return img, img_filename


In [7]:
EUROSAT_MEAN = [0.3444, 0.3803, 0.4078]
EUROSAT_STD  = [0.2025, 0.1366, 0.1148]
train_transforms = v2.Compose([
    v2.Resize((224,224), antialias=True),              # slightly larger before crop
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomApply([v2.RandomRotation(degrees=(90, 90))], p=0.5),   # only 90° multiples
    v2.RandomApply([v2.RandomRotation(degrees=(180, 180))], p=0.25),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD),
])

test_transforms = v2.Compose([
    v2.Resize((224, 224), antialias=True),   # ← same fix here
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD),
])
tta_transform = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    # Restrict to 90/180/270 rotations to prevent black zero-padding
    v2.RandomApply([v2.RandomRotation(degrees=(90, 90))], p=0.5),
    v2.RandomApply([v2.RandomRotation(degrees=(180, 180))], p=0.25),
    v2.RandomApply([v2.RandomRotation(degrees=(270, 270))], p=0.25),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD),
])

# 3. Now initialize your datasets
full_train_aug = EuroSATLabeled(img_dir=train_dir, transforms=train_transforms)
full_train_no_aug = EuroSATLabeled(img_dir=train_dir, transforms=test_transforms)
test_dataset = EuroSATUnlabeled(img_dir=test_dir, transforms=test_transforms)
#class for preprocessing of labelled data from train.zip

targets = [item[1] for item in full_train_aug.data]
indices = list(range(len(full_train_aug)))

train_indices, val_indices = train_test_split(
    indices, test_size=0.2, stratify=targets, random_state=42
)

train_dataset = Subset(full_train_aug, train_indices)
val_dataset = Subset(full_train_no_aug, val_indices)
test_dataset = EuroSATUnlabeled(
    img_dir=test_dir,
    transforms=test_transforms
)
# Validation TTA loader (uses same val_indices split)
tta_val_base    = EuroSATLabeled(img_dir=train_dir, transforms=tta_transform)
tta_val_dataset = Subset(tta_val_base, val_indices)
tta_val_loader  = DataLoader(tta_val_dataset, batch_size=256, shuffle=False,
                             num_workers=4, pin_memory=True, persistent_workers=True)
batch_size = 256
# Test TTA loader (for submission)
tta_test_dataset = EuroSATUnlabeled(img_dir=test_dir, transforms=tta_transform)
tta_test_loader  = DataLoader(tta_test_dataset, batch_size=256, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True)


In [8]:
batch_size = 256

# --- [UPDATED] Hardware Acceleration for DataLoaders ---
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True,persistent_workers=True)
val_dataloader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True   # ← was missing
)
test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Submission samples: {len(test_dataset)}")


Training samples: 18400
Validation samples: 4600
Submission samples: 4000


In [8]:
# import torch.nn.functional as F

# class BasicBlock(nn.Module):
#     expansion = 1
#     def __init__(self, in_channels, out_channels, stride=1):
#         super(BasicBlock, self).__init__()
#         self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(out_channels)
#         self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(out_channels)
#         self.shortcut = nn.Sequential()
#         if stride != 1 or in_channels != self.expansion * out_channels:
#             self.shortcut = nn.Sequential(
#                 nn.Conv2d(in_channels, self.expansion * out_channels, kernel_size=1, stride=stride, bias=False),
#                 nn.BatchNorm2d(self.expansion * out_channels)
#             )

#     def forward(self, x):
#         out = F.relu(self.bn1(self.conv1(x)))
#         out = self.bn2(self.conv2(out))
#         out += self.shortcut(x)
#         out = F.relu(out)
#         return out
# class Bottleneck(nn.Module):
#     expansion = 4 # This is crucial: Bottleneck increases out_channels by 4x

#     def __init__(self, in_channels, out_channels, stride=1):
#         super(Bottleneck, self).__init__()
#         # 1x1 conv to compress channels
#         self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(out_channels)
        
#         # 3x3 conv
#         self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(out_channels)
        
#         # 1x1 conv to expand channels back out
#         self.conv3 = nn.Conv2d(out_channels, self.expansion * out_channels, kernel_size=1, bias=False)
#         self.bn3 = nn.BatchNorm2d(self.expansion * out_channels)

#         self.shortcut = nn.Sequential()
#         if stride != 1 or in_channels != self.expansion * out_channels:
#             self.shortcut = nn.Sequential(
#                 nn.Conv2d(in_channels, self.expansion * out_channels, kernel_size=1, stride=stride, bias=False),
#                 nn.BatchNorm2d(self.expansion * out_channels)
#             )

#     def forward(self, x):
#         out = F.relu(self.bn1(self.conv1(x)))
#         out = F.relu(self.bn2(self.conv2(out)))
#         out = self.bn3(self.conv3(out))
#         out += self.shortcut(x)
#         out = F.relu(out)
#         return out

# class ResNet(nn.Module):
#     def __init__(self, block, num_blocks, num_classes=10):
#         super(ResNet, self).__init__()
#         self.in_channels = 64
#         self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(64)

#         self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
#         self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
#         self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
#         self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

#         self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
#         self.linear = nn.Linear(512 * block.expansion, num_classes)

#     def _make_layer(self, block, out_channels, num_blocks, stride):
#         strides = [stride] + [1] * (num_blocks - 1)
#         layers = []
#         for s in strides:
#             layers.append(block(self.in_channels, out_channels, s))
#             self.in_channels = out_channels * block.expansion
#         return nn.Sequential(*layers)

#     def forward(self, x):
#         out = F.relu(self.bn1(self.conv1(x)))
#         out = self.layer1(out)
#         out = self.layer2(out)
#         out = self.layer3(out)
#         out = self.layer4(out)
#         out = self.avgpool(out)
#         out = torch.flatten(out, 1)
#         out = self.linear(out)
#         return out

# def build_resnet50_64x64(num_classes):
#     # Change 'BasicBlock' to 'Bottleneck'
#     return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=num_classes)

# model = build_resnet50_64x64(num_classes=len(Class_Mapping)).to(device)

# def init_weights(m):
#     if isinstance(m, nn.Conv2d):
#         nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
#     elif isinstance(m, nn.BatchNorm2d):
#         nn.init.constant_(m.weight, 1)
#         nn.init.constant_(m.bias, 0)


# model.apply(init_weights)
# print("Applied Kaiming Normal initialization.")

In [10]:
import torchvision.models as models

# 1. Load the pretrained ResNet-18
# Using the updated weights API instead of the deprecated pretrained=True
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. Freeze all baseline convolutional layers
for param in model.parameters():
    param.requires_grad = False

# 3. Replace the final linear classification layer
# The 'fc' attribute in PyTorch ResNets is the final fully connected layer.
# By assigning a new nn.Linear module, it automatically has requires_grad=True.
num_ftrs = model.fc.in_features
print(num_ftrs)
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 1024),
    nn.BatchNorm1d(1024),
    nn.LeakyReLU(inplace=True),
    nn.Dropout(p=0.4),
    nn.Linear(1024,256),
    nn.BatchNorm1d(256),
    nn.ReLU(inplace=True),
    nn.Linear(256, len(Class_Mapping))
)


# Send the model to your configured device (GPU)
model = model.to(device)

print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 181MB/s] 


512
Total parameters: 11969354
Trainable parameters: 792842


In [12]:

learning_rate = 0.01                  # Step 6: was 0.001 — fresh linear layer needs a higher LR
early_stop_patience = 80              # Step 8: new

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)   # Step 7: was no smoothing

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=learning_rate,
    weight_decay=1e-3,
    amsgrad=True
)

num_epochs = 80
warmup_epochs = 5

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_epochs]
)

checkpoint_path = '/kaggle/working/EuroSAT_Build1_FrozenResNet18.pth'  # Step 11: was "ResNet50"
# model = torch.compile(model)
# print("Model successfully compiled and optimized.")

In [13]:
def run_tta(model, dataloader, device, n_aug=10, labeled=True):
    """
    Runs n_aug passes over the dataloader. Each pass applies a different
    random augmentation (since transforms are stochastic). Softmax probs
    are accumulated then averaged — only then is argmax taken.
    shuffle=False guarantees every pass is in the same sample order.
    """
    model.eval()
    n        = len(dataloader.dataset)
    acc_prob = torch.zeros(n, len(Class_Mapping))  # accumulated probabilities
    labels_out, names_out = [], []

    for pass_idx in range(n_aug):
        ptr = 0
        with torch.no_grad():
            if labeled:
                for X, y, _ in dataloader:
                    X = X.to(device)
                    with torch.amp.autocast('cuda'):
                        logits = model(X)
                    acc_prob[ptr:ptr + X.size(0)] += torch.softmax(logits, dim=1).cpu()
                    if pass_idx == 0:               # collect labels once
                        labels_out.extend(y.tolist())
                    ptr += X.size(0)
            else:
                for X, names in dataloader:
                    X = X.to(device)
                    with torch.amp.autocast('cuda'):
                        logits = model(X)
                    acc_prob[ptr:ptr + X.size(0)] += torch.softmax(logits, dim=1).cpu()
                    if pass_idx == 0:               # collect filenames once
                        names_out.extend(names)
                    ptr += X.size(0)

        print(f"  TTA pass {pass_idx + 1}/{n_aug} complete")

    avg_probs = acc_prob / n_aug                    # average BEFORE argmax — this is the key step
    preds     = avg_probs.argmax(dim=1).numpy()

    return (preds, labels_out) if labeled else (preds, names_out)

In [14]:
scaler = torch.amp.GradScaler('cuda')
# cutmix = v2.CutMix(num_classes=len(Class_Mapping), alpha=1.0)
cutmix = None  # ← Fix 1: was a bare `cutmix` expression → NameError. Define it explicitly.

def train_loop(dataloader, model, loss_fn, optimizer, cutmix=None):
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

    size = len(dataloader.dataset)
    num_batches = len(dataloader)   # ← needed for average loss
    correct = 0
    running_loss = 0.0              # ← Fix 2: accumulate loss across batches

    for batch, (X, y, _) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        if cutmix is not None:
            X, y = cutmix(X, y)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            pred = model(X)
            loss = loss_fn(pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()  # ← accumulate after scaler.update()

        if batch % 10 == 0:
            print(f"loss: {loss.item():>7f}  [{batch * dataloader.batch_size + len(X):>5d}/{size:>5d}]")

        if cutmix is not None:
            correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).sum().item()
        else:
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    train_accuracy = correct / size
    avg_train_loss = running_loss / num_batches  # ← average over all batches

    print(f"Training Accuracy: {100*train_accuracy:.1f}%  Avg Loss: {avg_train_loss:.6f}")
    return train_accuracy, avg_train_loss  # ← return both
def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y, _ in dataloader:
            X, y = X.to(device), y.to(device)
            with torch.amp.autocast('cuda'):
                pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    accuracy = correct / size
    print(f"Validation: Accuracy={100*accuracy:.1f}%, Avg Loss={test_loss:.6f}\n")
    return test_loss, accuracy   # ← was only returning test_loss

In [13]:
history = []
best_vacc = 0.0            # ← was best_vloss = float('inf')
early_stop_counter = 0
start_epoch = 0

for t in range(start_epoch, num_epochs):
    print(f"Epoch {t+1}")
    train_acc, train_loss = train_loop(train_dataloader, model, loss_fn, optimizer, cutmix=cutmix)
    vloss, vacc = test_loop(val_dataloader, model, loss_fn)

    scheduler.step()

    history.append({
        'epoch':      t + 1,
        'train_loss': train_loss,
        'train_acc':  train_acc,
        'val_loss':   vloss,
        'val_acc':    vacc,
        'lr':         scheduler.get_last_lr()[0]
    })

    if vacc > best_vacc:   # ← was: vloss < best_vloss
        print(f"Val accuracy improved {100*best_vacc:.2f}% → {100*vacc:.2f}%. Saving checkpoint...")
        best_vacc = vacc   # ← was: best_vloss = vloss
        early_stop_counter = 0
        torch.save({
            'epoch':           t + 1,
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_vacc':       best_vacc   # ← was: best_vloss
        }, checkpoint_path)
    else:
        early_stop_counter += 1
        print(f"No improvement. Early stop counter: {early_stop_counter}/{early_stop_patience}")
        if early_stop_counter >= early_stop_patience:
            print(f"Early stopping at epoch {t+1}. Best val accuracy: {100*best_vacc:.2f}%")
            break

print(f"\nTraining complete. Best val accuracy: {100*best_vacc:.2f}%")
pd.DataFrame(history)

Epoch 1
loss: 2.386451  [  256/18400]
loss: 0.792951  [ 2816/18400]
loss: 0.725031  [ 5376/18400]
loss: 0.661324  [ 7936/18400]
loss: 0.731970  [10496/18400]
loss: 0.629450  [13056/18400]
loss: 0.657999  [15616/18400]
loss: 0.664185  [18176/18400]
Training Accuracy: 90.0%  Avg Loss: 0.782704
Validation: Accuracy=95.0%, Avg Loss=0.649875

Val accuracy improved 0.00% → 95.00%. Saving checkpoint...
Epoch 2
loss: 0.658495  [  256/18400]
loss: 0.661564  [ 2816/18400]
loss: 0.687750  [ 5376/18400]
loss: 0.663626  [ 7936/18400]
loss: 0.681087  [10496/18400]
loss: 0.691272  [13056/18400]
loss: 0.662748  [15616/18400]
loss: 0.659017  [18176/18400]
Training Accuracy: 94.5%  Avg Loss: 0.664846
Validation: Accuracy=94.8%, Avg Loss=0.651774

No improvement. Early stop counter: 1/80
Epoch 3
loss: 0.640730  [  256/18400]
loss: 0.663177  [ 2816/18400]
loss: 0.637192  [ 5376/18400]
loss: 0.616811  [ 7936/18400]
loss: 0.672214  [10496/18400]
loss: 0.684520  [13056/18400]
loss: 0.621227  [15616/18400]
lo

,epoch,train_loss,train_acc,val_loss,val_acc,lr
0,1,0.782704,0.900380,0.649875,0.950000,0.002800
1,2,0.664846,0.944837,0.651774,0.947609,0.004600
2,3,0.651889,0.947663,0.643365,0.952391,0.006400
3,4,0.637571,0.952337,0.657661,0.943913,0.008200
4,5,0.634530,0.952989,0.649116,0.946522,0.010000
...,...,...,...,...,...,...
75,76,0.514647,0.998315,0.572591,0.970000,0.000071
76,77,0.514198,0.998913,0.572456,0.970870,0.000040
77,78,0.514383,0.998804,0.572591,0.971522,0.000019
78,79,0.514536,0.998587,0.572673,0.970870,0.000005


In [20]:
# --- Load best checkpoint ---
print("Loading best checkpoint...")
model_path = "/kaggle/input/models/sakshamgarg2323/eurosat/pytorch/default/1/EuroSAT_Build1_FrozenResNet18.pth"

# 1. Load the checkpoint file
checkpoint = torch.load(model_path, map_location=device)

# 2. Extract the weights using the correct key (try 'model_state' or 'state_dict')
# If you get a KeyError here, change 'model_state' to 'state_dict'
model.load_state_dict(checkpoint['model_state'])
# --- Step 4: TTA Validation ---
print("\nRunning TTA on validation set...")
tta_preds, true_labels = run_tta(model, tta_val_loader, device, n_aug=15, labeled=True)

tta_acc = (tta_preds == np.array(true_labels)).mean()
print(f"\nTTA Validation Accuracy: {100 * tta_acc:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, tta_preds))
print("\nClassification Report:")
print(classification_report(true_labels, tta_preds, target_names=list(Class_Mapping.keys())))

# --- Step 5: TTA Test Submission ---
print("\nRunning TTA on test set...")
tta_test_preds, filenames = run_tta(model, tta_test_loader, device, n_aug=15, labeled=False)

submission_df = pd.DataFrame({'image_id': filenames, 'label': tta_test_preds})
submission_df = submission_df.sort_values('image_id').reset_index(drop=True)
submission_df.to_csv('2025MT11352_CNN.csv', index=False)
print(f"Submission saved — {len(submission_df)} predictions.")

Loading best checkpoint...

Running TTA on validation set...
  TTA pass 1/15 complete
  TTA pass 2/15 complete
  TTA pass 3/15 complete
  TTA pass 4/15 complete
  TTA pass 5/15 complete
  TTA pass 6/15 complete
  TTA pass 7/15 complete
  TTA pass 8/15 complete
  TTA pass 9/15 complete
  TTA pass 10/15 complete
  TTA pass 11/15 complete
  TTA pass 12/15 complete
  TTA pass 13/15 complete
  TTA pass 14/15 complete
  TTA pass 15/15 complete

TTA Validation Accuracy: 97.85%

Confusion Matrix:
[[508   0   1   0   0   4   5   0   2   0]
 [  0 520   0   0   0   0   0   0   0   0]
 [  0   1 511   1   0   4   2   1   0   0]
 [  2   0   1 406   0   0   4   2   5   0]
 [  0   0   0   3 413   0   0   4   0   0]
 [  5   0   3   3   0 307   1   0   1   0]
 [  7   0   9   1   1   1 400   1   0   0]
 [  0   0   0   1   4   0   0 515   0   0]
 [  2   0   0  13   1   2   0   0 402   0]
 [  0   0   1   0   0   0   0   0   0 519]]

Classification Report:
                      precision    recall  f1-score